# MultiQubitEntanglementV0

## Description

`MultiQubitEntanglementV0` is a single-qubit quantum environment intended for teaching, visualization, and 
simple RL-style control experiments. It is based on the `QuantumEnv` base class. The environment allows an agent to evolve multiple qubits with discrete quantum gates to reach a target entangled state, such as a GHZ or Bell state.

The agent's goal is to steer the qubits from the initial all-zero state ∣0…0⟩ to a specified target entangled state within a limited number of steps.

The **Render** function visualizes the amplitude probabilities of computational basis states across steps, giving an intuitive picture of state evolution.

## Action Space

- The action space is **discrete**, where each integer corresponds to a quantum gate 
applied to one or two qubits. The set of available actions includes:

| Num | Action         | Description                                               |
| --- | -------------- | --------------------------------------------------------- |
| 0   | `H_i`          | Hadamard gate on qubit i                                  |
| 1   | `X_i`          | Pauli-X (NOT) on qubit i                                  |
| 2   | `Y_i`          | Pauli-Y on qubit i                                        |
| 3   | `Z_i`          | Pauli-Z on qubit i                                        |
| 4   | `S_i`          | Phase gate (S) on qubit i                                 |
| 5   | `SDG_i`        | S† (S dagger) on qubit i                                  |
| 6   | `T_i`          | T gate on qubit i                                         |
| 7   | `TDG_i`        | T† (T dagger) on qubit i                                  |
| 8   | `RX_i_pi_2`    | Rotation about X by +π/2 on qubit i                       |
| 9   | `RX_i_pi_4`    | Rotation about X by +π/4 on qubit i                       |
| 10  | `RX_i_-pi_4`   | Rotation about X by -π/4 on qubit i                       |
| 11  | `RY_i_pi_2`    | Rotation about Y by +π/2 on qubit i                       |
| 12  | `RY_i_pi_4`    | Rotation about Y by +π/4 on qubit i                       |
| 13  | `RY_i_-pi_4`   | Rotation about Y by -π/4 on qubit i                       |
| 14  | `RZ_i_pi_2`    | Rotation about Z by +π/2 on qubit i                       |
| 15  | `RZ_i_pi_4`    | Rotation about Z by +π/4 on qubit i                       |
| 16  | `RZ_i_-pi_4`   | Rotation about Z by -π/4 on qubit i                       |
| 17  | `CNOT_i_j`     | Controlled-NOT gate with controlqubit i and target qubit j|

- `i` and `j` are qubit indices ranging from 0 to `n_qubits - 1`.
- The `action_space` is `spaces.Discrete(len(actions))`.

## Observation Space

Observations are **concatenated real and imaginary parts** of the full statevector:

```python
obs = np.concatenate([np.real(state), np.imag(state)])
```

- Shape: `(2**n_qubits * 2,)`
- Dtype: `float32`
- Each element corresponds to the real or imaginary part of a computational basis amplitude.
- This representation allows RL agents to process the quantum state as a vector of continuous values.

## Rewards

Rewards are based on the **fidelity** between the current quantum state and the target state:

```python
reward = |⟨target|state⟩|^2
```

- Continuous reward in `[0, 1]`.
- Higher reward means the current state is closer to the target entangled state.
- An episode terminates when:
    1. **Success:** `reward >= reward_tolerance` (default 0.99)
    2. **Truncation:** `steps >= max_steps` (default 20)

## Reset Behavior

On `reset()`:

- `steps` is set to 0
- The quantum state is initialized to |0...0⟩
- History is initialized with the current observation
- Returns the first observation and an empty `info` dict

Optionally, the target state can be customized by passing a `target_state` argument
to the constructor; otherwise, the default is the **GHZ-like superposition**:

```text
|target⟩ = (|0...0⟩ + |1...1⟩) / √2
```

## Step Behavior

`step(action)`:

1. Applies the specified gate to the current quantum state.
2. Updates the observation (`obs`) with the new statevector.
3. Calculates the reward (fidelity with the target state).
4. Increments the step counter.
5. Checks for `done` conditions (reward threshold or max steps).
6. Appends the observation to `history`.
7. Returns `(obs, reward, done, info)` as per Gymnasium convention.

In [ ]:
import numpy as np
from qrl.env import MultiQubitEntanglementV0

# Target: 2-qubit Bell state (|00> + |11>)/sqrt(2)
target_state = np.array([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)], dtype=complex)

# Initialize environment
env = MultiQubitEntanglementV0(n_qubits=2, target_state=target_state, max_steps=20, reward_tolerance=0.99)

# Reset environment
obs, _ = env.reset()
print("Initial Observation (real + imag parts):", obs)

# Sample random actions
for _ in range(env.max_steps):
    action = env.action_space.sample()
    obs, reward, done, _ = env.step(action)
    print(f"Action {action} ({env.actions[action]}) -> Reward: {reward:.4f} Done: {done}")

    if done:
        break

# Render amplitude probabilities
env.render(save_path="bell_state")


## Notes & Extensions

- Currently supports **pure states** represented as statevectors.
- Could be extended to **mixed states** or include **noise channels**.
- Reward shaping or step penalties can encourage faster entanglement.
- Observation augmentation (step number, recent gate, fidelity) can improve RL training.

## Version History

* **v0**: Initial design and implementation. Multi-qubit entanglement environment with fidelity reward, step tracking, and amplitude visualization.

## References:

* Nielsen & Chuang, Quantum Computation and Quantum Information
* Qiskit / PennyLane tutorials on multi-qubit gates and GHZ/Bell states
